In [ ]:
# Setup and imports
import sys
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add src to path for imports
sys.path.append('../src')

from features.hedge_pressure_research import (
    compute_hedge_pressure,
    validate_hedge_pressure_signal,
    generate_synthetic_hedge_data,
    analyze_hedge_pressure_patterns,
    HedgePressureConfig,
    HedgePressureResult,
)

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

print("Hedge Pressure Research Experiment Setup Complete")
print(f"Current timestamp: {datetime.now().isoformat()}")

## Signal Verification

Test the core hedge pressure computation with known scenarios.

In [ ]:
# Test basic functionality
print("=== Signal Verification ===")

# Test 1: Balanced market (no pressure)
calls_balanced = {18000: 1000, 18500: 800, 19000: 600}
puts_balanced = {18000: 1000, 18500: 800, 19000: 600}
spot_price = 18500

result_balanced = compute_hedge_pressure(calls_balanced, puts_balanced, spot_price)
print(f"Balanced market - Pressure: {result_balanced.pressure:.4f}, Confidence: {result_balanced.confidence:.4f}")

# Test 2: Bullish bias (more calls)
calls_bullish = {18000: 1500, 18500: 1200, 19000: 900}
puts_bullish = {18000: 500, 18500: 400, 19000: 300}

result_bullish = compute_hedge_pressure(calls_bullish, puts_bullish, spot_price)
print(f"Bullish bias - Pressure: {result_bullish.pressure:.4f}, Confidence: {result_bullish.confidence:.4f}")

# Test 3: Bearish bias (more puts)
calls_bearish = {18000: 500, 18500: 400, 19000: 300}
puts_bearish = {18000: 1500, 18500: 1200, 19000: 900}

result_bearish = compute_hedge_pressure(calls_bearish, puts_bearish, spot_price)
print(f"Bearish bias - Pressure: {result_bearish.pressure:.4f}, Confidence: {result_bearish.confidence:.4f}")

# Test 4: Extreme imbalance
calls_extreme = {18000: 2000, 18500: 1800, 19000: 1500}
puts_extreme = {18000: 100, 18500: 50, 19000: 25}

result_extreme = compute_hedge_pressure(calls_extreme, puts_extreme, spot_price)
print(f"Extreme bullish - Pressure: {result_extreme.pressure:.4f}, Confidence: {result_extreme.confidence:.4f}")

print("\nSignal verification complete.")

## Synthetic Data Testing

Generate synthetic data to test signal behavior across different scenarios.

In [ ]:
# Generate synthetic data for testing
print("=== Synthetic Data Generation ===")

# Base OI levels
base_calls = {18000: 1000, 18200: 800, 18400: 600, 18600: 500, 18800: 400, 19000: 300}
base_puts = {18000: 1000, 18200: 800, 18400: 600, 18600: 500, 18800: 400, 19000: 300}
spot_price = 18500

# Test different pressure levels
pressure_levels = [-1.0, -0.5, 0.0, 0.5, 1.0]
synthetic_results = []

for pressure in pressure_levels:
    syn_calls, syn_puts = generate_synthetic_hedge_data(
        base_calls, base_puts, spot_price, pressure
    )
    
    result = compute_hedge_pressure(syn_calls, syn_puts, spot_price)
    
    synthetic_results.append({
        'target_pressure': pressure,
        'computed_pressure': result.pressure,
        'imbalance_ratio': result.imbalance_ratio,
        'confidence': result.confidence,
        'call_total': result.call_oi_total,
        'put_total': result.put_oi_total,
    })
    
    print(f"Target: {pressure:.1f} -> Computed: {result.pressure:.4f} (Imbalance: {result.imbalance_ratio:.4f})")

# Convert to DataFrame for analysis
df_synthetic = pd.DataFrame(synthetic_results)
print(f"\nSynthetic data generation complete. Generated {len(synthetic_results)} test cases.")

In [ ]:
# Visualize synthetic data results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Pressure correlation
axes[0,0].scatter(df_synthetic['target_pressure'], df_synthetic['computed_pressure'], alpha=0.7)
axes[0,0].plot([-1, 1], [-1, 1], 'r--', alpha=0.5)
axes[0,0].set_xlabel('Target Pressure')
axes[0,0].set_ylabel('Computed Pressure')
axes[0,0].set_title('Pressure Correlation')
axes[0,0].grid(True, alpha=0.3)

# Imbalance vs Pressure
axes[0,1].scatter(df_synthetic['imbalance_ratio'], df_synthetic['computed_pressure'], alpha=0.7)
axes[0,1].set_xlabel('Imbalance Ratio')
axes[0,1].set_ylabel('Computed Pressure')
axes[0,1].set_title('Imbalance vs Pressure')
axes[0,1].grid(True, alpha=0.3)

# Confidence distribution
axes[1,0].bar(range(len(df_synthetic)), df_synthetic['confidence'], alpha=0.7)
axes[1,0].set_xlabel('Test Case')
axes[1,0].set_ylabel('Confidence')
axes[1,0].set_title('Confidence by Test Case')
axes[1,0].set_xticks(range(len(df_synthetic)))
axes[1,0].set_xticklabels([f'{x:.1f}' for x in df_synthetic['target_pressure']])

# OI totals
axes[1,1].bar(range(len(df_synthetic)), df_synthetic['call_total'], alpha=0.7, label='Calls')
axes[1,1].bar(range(len(df_synthetic)), df_synthetic['put_total'], alpha=0.7, 
         bottom=df_synthetic['call_total'], label='Puts')
axes[1,1].set_xlabel('Test Case')
axes[1,1].set_ylabel('OI Total')
axes[1,1].set_title('Call/Put OI Distribution')
axes[1,1].set_xticks(range(len(df_synthetic)))
axes[1,1].set_xticklabels([f'{x:.1f}' for x in df_synthetic['target_pressure']])
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Statistical analysis
correlation = np.corrcoef(df_synthetic['target_pressure'], df_synthetic['computed_pressure'])[0,1]
print(f"Pressure correlation: {correlation:.4f}")
print(f"Mean confidence: {df_synthetic['confidence'].mean():.4f}")
print(f"Pressure range: [{df_synthetic['computed_pressure'].min():.4f}, {df_synthetic['computed_pressure'].max():.4f}]")

## Statistical Properties Analysis

Analyze the statistical properties of the hedge pressure signal.

In [ ]:
# Statistical analysis of signal properties
print("=== Statistical Properties Analysis ===")

# Generate large sample for statistical testing
np.random.seed(42)  # For reproducibility
n_samples = 1000

statistical_results = []
for i in range(n_samples):
    # Random pressure level
    pressure_level = np.random.uniform(-1, 1)
    
    # Generate synthetic data
    syn_calls, syn_puts = generate_synthetic_hedge_data(
        base_calls, base_puts, spot_price, pressure_level
    )
    
    # Compute signal
    result = compute_hedge_pressure(syn_calls, syn_puts, spot_price)
    
    statistical_results.append({
        'input_pressure': pressure_level,
        'output_pressure': result.pressure,
        'imbalance_ratio': result.imbalance_ratio,
        'confidence': result.confidence,
        'error': abs(result.pressure - pressure_level),
    })

df_stats = pd.DataFrame(statistical_results)

# Basic statistics
print(f"Sample size: {len(df_stats)}")
print(f"Mean error: {df_stats['error'].mean():.4f}")
print(f"Std error: {df_stats['error'].std():.4f}")
print(f"Max error: {df_stats['error'].max():.4f}")

# Distribution analysis
print(f"\nOutput pressure distribution:")
print(f"  Mean: {df_stats['output_pressure'].mean():.4f}")
print(f"  Std: {df_stats['output_pressure'].std():.4f}")
print(f"  Skew: {df_stats['output_pressure'].skew():.4f}")
print(f"  Kurtosis: {df_stats['output_pressure'].kurtosis():.4f}")

# Test for normality
stat, p_value = stats.shapiro(df_stats['output_pressure'].sample(min(5000, len(df_stats))))
print(f"\nNormality test (Shapiro-Wilk): p-value = {p_value:.6f}")
print(f"Distribution appears {'normal' if p_value > 0.05 else 'non-normal'}")

# Correlation analysis
correlation = df_stats['input_pressure'].corr(df_stats['output_pressure'])
print(f"\nInput-output correlation: {correlation:.4f}")

# Confidence analysis
print(f"Confidence distribution:")
print(f"  Mean: {df_stats['confidence'].mean():.4f}")
print(f"  Std: {df_stats['confidence'].std():.4f}")
print(f"  Min: {df_stats['confidence'].min():.4f}")
print(f"  Max: {df_stats['confidence'].max():.4f}")

In [ ]:
# Visualize statistical properties
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Error distribution
axes[0,0].hist(df_stats['error'], bins=50, alpha=0.7, edgecolor='black')
axes[0,0].set_xlabel('Absolute Error')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title('Error Distribution')
axes[0,0].axvline(df_stats['error'].mean(), color='red', linestyle='--', label=f'Mean: {df_stats["error"].mean():.4f}')
axes[0,0].legend()

# Pressure distribution
axes[0,1].hist(df_stats['output_pressure'], bins=50, alpha=0.7, edgecolor='black')
axes[0,1].set_xlabel('Output Pressure')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('Pressure Distribution')

# Confidence distribution
axes[0,2].hist(df_stats['confidence'], bins=50, alpha=0.7, edgecolor='black')
axes[0,2].set_xlabel('Confidence')
axes[0,2].set_ylabel('Frequency')
axes[0,2].set_title('Confidence Distribution')

# Q-Q plot for normality
stats.probplot(df_stats['output_pressure'], dist="norm", plot=axes[1,0])
axes[1,0].set_title('Q-Q Plot (Normality Test)')

# Error vs Input pressure
axes[1,1].scatter(df_stats['input_pressure'], df_stats['error'], alpha=0.1)
axes[1,1].set_xlabel('Input Pressure')
axes[1,1].set_ylabel('Absolute Error')
axes[1,1].set_title('Error vs Input Pressure')

# Confidence vs Error
axes[1,2].scatter(df_stats['confidence'], df_stats['error'], alpha=0.1)
axes[1,2].set_xlabel('Confidence')
axes[1,2].set_ylabel('Absolute Error')
axes[1,2].set_title('Error vs Confidence')

plt.tight_layout()
plt.show()

## Failure Mode Testing

Test the signal under various failure conditions and edge cases.

In [ ]:
# Failure mode testing
print("=== Failure Mode Testing ===")

failure_tests = [
    {
        'name': 'Insufficient strikes',
        'calls': {18500: 100},
        'puts': {18500: 100},
        'spot': 18500,
        'expected_error': 'insufficient_strikes',
    },
    {
        'name': 'Zero OI',
        'calls': {18000: 0, 18500: 0, 19000: 0},
        'puts': {18000: 0, 18500: 0, 19000: 0},
        'spot': 18500,
        'expected_error': 'insufficient_liquidity',
    },
    {
        'name': 'Very low liquidity',
        'calls': {18000: 10, 18500: 5, 19000: 2},
        'puts': {18000: 10, 18500: 5, 19000: 2},
        'spot': 18500,
        'expected_error': 'insufficient_liquidity',
    },
    {
        'name': 'Extreme imbalance',
        'calls': {18000: 10000, 18500: 5000, 19000: 1000},
        'puts': {18000: 10, 18500: 5, 19000: 1},
        'spot': 18500,
        'expected_pressure': 1.0,  # Should be clamped
    },
    {
        'name': 'Invalid spot price',
        'calls': {18000: 1000, 18500: 800, 19000: 600},
        'puts': {18000: 1000, 18500: 800, 19000: 600},
        'spot': -100,
        'expected_exception': ValueError,
    },
]

failure_results = []
for test in failure_tests:
    try:
        result = compute_hedge_pressure(test['calls'], test['puts'], test['spot'])
        
        # Check for expected error
        has_expected_error = (
            'expected_error' in test and 
            result.metadata.get('error') == test['expected_error']
        )
        
        # Check for expected pressure
        has_expected_pressure = (
            'expected_pressure' in test and 
            abs(result.pressure - test['expected_pressure']) < 0.01
        )
        
        failure_results.append({
            'test': test['name'],
            'success': has_expected_error or has_expected_pressure,
            'pressure': result.pressure,
            'confidence': result.confidence,
            'error': result.metadata.get('error'),
            'exception': None,
        })
        
    except Exception as e:
        expected_exception = test.get('expected_exception')
        success = expected_exception and isinstance(e, expected_exception)
        
        failure_results.append({
            'test': test['name'],
            'success': success,
            'pressure': None,
            'confidence': None,
            'error': None,
            'exception': str(type(e).__name__),
        })

# Display results
df_failures = pd.DataFrame(failure_results)
print("Failure mode test results:")
for _, row in df_failures.iterrows():
    status = "✓" if row['success'] else "✗"
    print(f"{status} {row['test']}: {row['exception'] or row['error'] or 'OK'}")

success_rate = df_failures['success'].mean()
print(f"\nOverall success rate: {success_rate:.1%}")

## Validation Framework Testing

Test the validation functions and regime compatibility.

In [ ]:
# Validation framework testing
print("=== Validation Framework Testing ===")

# Test different signal qualities
validation_tests = [
    {
        'name': 'High quality signal',
        'calls': {18000: 2000, 18500: 1500, 19000: 1000},
        'puts': {18000: 500, 18500: 300, 19000: 200},
        'spot': 18500,
        'regime': None,
    },
    {
        'name': 'Low confidence signal',
        'calls': {18000: 50, 18500: 30, 19000: 20},
        'puts': {18000: 50, 18500: 30, 19000: 20},
        'spot': 18500,
        'regime': None,
    },
    {
        'name': 'Expiry regime',
        'calls': {18000: 1000, 18500: 800, 19000: 600},
        'puts': {18000: 1000, 18500: 800, 19000: 600},
        'spot': 18500,
        'regime': {'regime': 'expiry'},
    },
    {
        'name': 'Earnings regime',
        'calls': {18000: 1000, 18500: 800, 19000: 600},
        'puts': {18000: 1000, 18500: 800, 19000: 600},
        'spot': 18500,
        'regime': {'regime': 'earnings'},
    },
]

validation_results = []
for test in validation_tests:
    result = compute_hedge_pressure(test['calls'], test['puts'], test['spot'])
    validation = validate_hedge_pressure_signal(result, test['regime'])
    
    validation_results.append({
        'test': test['name'],
        'is_valid': validation['is_valid'],
        'confidence_assessment': validation['confidence_assessment'],
        'regime_compatibility': validation['regime_compatibility'],
        'warning_count': len(validation['warnings']),
        'warnings': validation['warnings'],
        'pressure': result.pressure,
        'confidence': result.confidence,
    })

# Display validation results
df_validation = pd.DataFrame(validation_results)
print("Validation test results:")
for _, row in df_validation.iterrows():
    print(f"{row['test']}:")
    print(f"  Valid: {row['is_valid']}, Confidence: {row['confidence_assessment']}, Regime: {row['regime_compatibility']}")
    print(f"  Warnings: {row['warning_count']} - {row['warnings'][:2]}..." if len(row['warnings']) > 2 else f"  Warnings: {row['warnings']}")
    print(f"  Pressure: {row['pressure']:.4f}, Confidence: {row['confidence']:.4f}")
    print()

# Summary statistics
valid_signals = df_validation['is_valid'].sum()
total_signals = len(df_validation)
print(f"Validation summary: {valid_signals}/{total_signals} signals passed validation")
print(f"Average warnings per signal: {df_validation['warning_count'].mean():.1f}")

## Walk-Forward Validation Setup

Set up the framework for walk-forward validation (to be run with real historical data).

In [ ]:
# Walk-forward validation setup
print("=== Walk-Forward Validation Setup ===")

# Generate synthetic time series for validation
np.random.seed(123)
n_periods = 100
timestamps = [datetime(2024, 1, 1) + timedelta(hours=i) for i in range(n_periods)]

# Simulate evolving market conditions
calls_history = []
puts_history = []
spot_prices = []

# Start with balanced market
current_calls = {18000: 1000, 18200: 800, 18400: 600, 18600: 500, 18800: 400, 19000: 300}
current_puts = {18000: 1000, 18200: 800, 18400: 600, 18600: 500, 18800: 400, 19000: 300}
current_spot = 18500

# Simulate market evolution with random shocks
for i in range(n_periods):
    # Add some trend and noise to spot price
    spot_noise = np.random.normal(0, 50)
    current_spot += spot_noise
    current_spot = max(17000, min(20000, current_spot))  # Keep in range
    
    # Generate pressure with some persistence
    if i == 0:
        pressure = np.random.normal(0, 0.3)
    else:
        # Add persistence and random shocks
        pressure = 0.7 * pressure + 0.3 * np.random.normal(0, 0.3)
    
    pressure = max(-1, min(1, pressure))  # Clamp
    
    # Generate OI based on pressure
    syn_calls, syn_puts = generate_synthetic_hedge_data(
        current_calls, current_puts, current_spot, pressure
    )
    
    # Add some evolution to base OI
    for strike in current_calls:
        current_calls[strike] = max(100, int(current_calls[strike] * (1 + np.random.normal(0, 0.02))))
        current_puts[strike] = max(100, int(current_puts[strike] * (1 + np.random.normal(0, 0.02))))
    
    calls_history.append(syn_calls.copy())
    puts_history.append(syn_puts.copy())
    spot_prices.append(current_spot)

# Run analysis
analysis = analyze_hedge_pressure_patterns(calls_history, puts_history, spot_prices, timestamps)

print(f"Generated {n_periods} periods of synthetic data")
print(f"Pressure stats: mean={analysis['pressure_stats']['mean']:.4f}, std={analysis['pressure_stats']['std']:.4f}")
print(f"Confidence stats: mean={analysis['confidence_stats']['mean']:.4f}, std={analysis['confidence_stats']['std']:.4f}")
print(f"Imbalance stats: mean={analysis['imbalance_stats']['mean']:.4f}, std={analysis['imbalance_stats']['std']:.4f}")

# Convert results to DataFrame for plotting
results_df = pd.DataFrame([
    {
        'timestamp': r['timestamp'],
        'pressure': r['result'].pressure,
        'confidence': r['result'].confidence,
        'imbalance_ratio': r['result'].imbalance_ratio,
        'spot_price': r['spot_price'],
    }
    for r in analysis['results']
])

print("\nWalk-forward validation data prepared.")
print("To run with real data, replace synthetic generation with historical OI data.")

In [ ]:
# Visualize walk-forward results
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

# Pressure over time
axes[0].plot(results_df['timestamp'], results_df['pressure'], 'b-', alpha=0.7, linewidth=2)
axes[0].fill_between(results_df['timestamp'], 
                    results_df['pressure'] - results_df['confidence'],
                    results_df['pressure'] + results_df['confidence'], 
                    alpha=0.2, color='blue')
axes[0].set_ylabel('Hedge Pressure')
axes[0].set_title('Hedge Pressure Over Time (with confidence bands)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Confidence over time
axes[1].plot(results_df['timestamp'], results_df['confidence'], 'g-', alpha=0.7)
axes[1].set_ylabel('Confidence')
axes[1].set_title('Signal Confidence Over Time')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

# Spot price and imbalance
ax2_twin = axes[2].twinx()
line1 = axes[2].plot(results_df['timestamp'], results_df['spot_price'], 'r-', alpha=0.7, label='Spot Price')
line2 = ax2_twin.plot(results_df['timestamp'], results_df['imbalance_ratio'], 'purple', alpha=0.7, label='Imbalance Ratio')
axes[2].set_ylabel('Spot Price', color='red')
ax2_twin.set_ylabel('Imbalance Ratio', color='purple')
axes[2].set_title('Spot Price and Imbalance Ratio Over Time')
axes[2].grid(True, alpha=0.3)

# Combined legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
axes[2].legend(lines, labels, loc='upper left')

plt.tight_layout()
plt.show()

# Statistical analysis of time series
print("\nTime series analysis:")
print(f"Pressure autocorrelation (lag 1): {results_df['pressure'].autocorr(lag=1):.4f}")
print(f"Pressure volatility: {results_df['pressure'].std():.4f}")
print(f"Confidence stability: {results_df['confidence'].std():.4f}")
print(f"Extreme pressure events (>0.5): {(abs(results_df['pressure']) > 0.5).sum()}")
print(f"Low confidence periods (<0.3): {(results_df['confidence'] < 0.3).sum()}")

## Experiment Summary and Next Steps

Summarize the hedge pressure signal research findings and outline next steps.

In [ ]:
# Experiment summary
print("=== Hedge Pressure Signal Research Summary ===")
print(f"Experiment completed: {datetime.now().isoformat()}")
print()

# Key findings
print("KEY FINDINGS:")
print(f"• Signal implementation: {'✓ Complete' if True else '✗ Incomplete'}")
print(f"• Test coverage: {len(df_stats)} statistical tests")
print(f"• Failure mode testing: {success_rate:.1%} success rate")
print(f"• Validation framework: {valid_signals}/{total_signals} signals validated")
print(f"• Synthetic data correlation: {correlation:.4f}")
print(f"• Walk-forward periods: {n_periods}")
print()

# Signal characteristics
print("SIGNAL CHARACTERISTICS:")
print(f"• Range: [{df_stats['output_pressure'].min():.4f}, {df_stats['output_pressure'].max():.4f}]")
print(f"• Mean error: {df_stats['error'].mean():.4f}")
print(f"• Distribution: {'Normal' if p_value > 0.05 else 'Non-normal'}")
print(f"• Confidence range: [{df_stats['confidence'].min():.4f}, {df_stats['confidence'].max():.4f}]")
print()

# Validation status
validation_status = "READY" if (
    correlation > 0.8 and 
    success_rate > 0.9 and 
    valid_signals / total_signals > 0.7
) else "NEEDS_WORK"

print(f"VALIDATION STATUS: {validation_status}")
print()

# Next steps
print("NEXT STEPS:")
if validation_status == "READY":
    print("• ✓ Proceed to PROMO-2025-12-002: Hedge pressure promotion checklist")
    print("• ✓ Ready for integration into intent engine")
    print("• ✓ Can be used for backtesting with historical data")
else:
    print("• ✗ Address validation issues")
    print("• ✗ Improve signal accuracy")
    print("• ✗ Enhance failure mode handling")
    print("• ✗ Re-run statistical tests")

print("• → EXP-2025-12-003: Implement basis pressure signal research")
print()

print("EXPERIMENT COMPLETE")
print("Files created:")
print("• research/python/src/features/hedge_pressure_research.py")
print("• research/python/notebooks/exploratory/hedge_pressure_experiment.ipynb")
print("• research/python/tests/test_hedge_pressure_research.py")